In [ ]:
# pip install datasets spacy networkx scikit-learn
# python -m spacy download en_core_web_sm

import spacy
import networkx as nx
import numpy as np
from datasets import load_dataset
from sklearn.metrics import average_precision_score, roc_auc_score
from collections import defaultdict

nlp = spacy.load("en_core_web_sm")



In [ ]:
# ── 1. LOAD DATA ──────────────────────────────────────────────────────────────

dataset = load_dataset("hotpot_qa", "distractor", split="validation[:500]")
# 'distractor' split includes 10 paragraphs per example (2 gold + 8 noise)
# We use the validation set to avoid downloading the full train corpus



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

distractor/train-00000-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/train-00001-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/validation-00000-of-00001.par(…):   0%|          | 0.00/27.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

In [ ]:
dataset

Dataset({
    features: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'],
    num_rows: 500
})

In [ ]:
dataset['question'][0]

'Were Scott Derrickson and Ed Wood of the same nationality?'

In [ ]:
dataset['answer'][0]

'yes'

In [ ]:
dataset['context'][0]

{'title': ['Ed Wood (film)',
  'Scott Derrickson',
  'Woodson, Arkansas',
  'Tyler Bates',
  'Ed Wood',
  'Deliver Us from Evil (2014 film)',
  'Adam Collis',
  'Sinister (film)',
  'Conrad Brooks',
  'Doctor Strange (2016 film)'],
 'sentences': [['Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood.',
   " The film concerns the period in Wood's life when he made his best-known films as well as his relationship with actor Bela Lugosi, played by Martin Landau.",
   ' Sarah Jessica Parker, Patricia Arquette, Jeffrey Jones, Lisa Marie, and Bill Murray are among the supporting cast.'],
  ['Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer.',
   ' He lives in Los Angeles, California.',
   ' He is best known for directing horror films such as "Sinister", "The Exorcism of Emily Rose", and "Deliver Us From Evil", as well as the 2016 Marvel Cinematic Univer

In [ ]:
dataset['supporting_facts'][0]

{'title': ['Scott Derrickson', 'Ed Wood'], 'sent_id': [0, 0]}

In [ ]:
# ── 2. ENTITY EXTRACTION ─────────────────────────────────────────────────────

def extract_entities(text: str) -> list[str]:
    doc = nlp(text)
    return [ent.text.lower().strip() for ent in doc.ents
            if ent.label_ in {"PERSON", "ORG", "GPE", "LOC", "WORK_OF_ART", "EVENT"}]

def build_example_graph(example: dict) -> nx.Graph:
    """
    Build a co-occurrence graph for a single HotpotQA example.
    Nodes = entities. Edge (u,v) exists if u and v appear in the same sentence.
    Edge weight = number of sentences they co-occur in.
    """
    G = nx.Graph()
    context = example["context"]  # {"title": [...], "sentences": [[...]]}

    for title, sentences in zip(context["title"], context["sentences"]):
        for sentence in sentences:
            ents = extract_entities(sentence)
            ents = list(set(ents))  # deduplicate within sentence
            for i in range(len(ents)):
                for j in range(i + 1, len(ents)):
                    u, v = ents[i], ents[j]
                    if G.has_edge(u, v):
                        G[u][v]["weight"] += 1
                    else:
                        G.add_edge(u, v, weight=1)
    return G



In [ ]:
# ── 3. PERSONALIZED PAGERANK AS LINK SCORER ───────────────────────────────────

def score_candidates_ppr(G: nx.Graph, question_entities: list[str],
                          alpha: float = 0.85) -> dict[str, float]:
    """
    Run Personalized PageRank seeded on question entities.
    Returns a score dict over all nodes — higher = more relevant to the question.
    This is the key insight: PPR propagates 'relevance' through the graph
    from your seed nodes, which is exactly what Graph RAG retrieval does.
    """
    if not question_entities or G.number_of_nodes() == 0:
        return {}

    # Seed distribution: uniform over question entities that exist in graph
    seeds = [e for e in question_entities if e in G.nodes]
    if not seeds:
        return {}

    personalization = {node: 0.0 for node in G.nodes}
    for s in seeds:
        personalization[s] = 1.0 / len(seeds)

    scores = nx.pagerank(G, alpha=alpha, personalization=personalization,
                         weight="weight")
    return scores



In [ ]:
# ── 4. COMPARISON HEURISTICS ──────────────────────────────────────────────────

def score_candidates_jaccard(G: nx.Graph, question_entities: list[str]) -> dict[str, float]:
    scores = defaultdict(float)
    seeds = [e for e in question_entities if e in G.nodes]
    for seed in seeds:
        for candidate in G.nodes:
            if candidate in seeds:
                continue
            try:
                j = list(nx.jaccard_coefficient(G, [(seed, candidate)]))[0][2]
                scores[candidate] = max(scores[candidate], j)
            except Exception:
                pass
    return dict(scores)

def score_candidates_adamic(G: nx.Graph, question_entities: list[str]) -> dict[str, float]:
    scores = defaultdict(float)
    seeds = [e for e in question_entities if e in G.nodes]
    for seed in seeds:
        for u, v, s in nx.adamic_adar_index(G, [(seed, c) for c in G.nodes if c not in seeds]):
            scores[v] = max(scores[v], s)
    return dict(scores)

def score_candidates_katz(G: nx.Graph, question_entities: list[str],
                           beta: float = 0.05, max_path: int = 3) -> dict[str, float]:
    """
    Katz index: sum over paths of length 1..max_path, decayed by beta^length.
    We compute this via repeated matrix-vector multiplication — tractable for
    small per-example graphs (typically <200 nodes).
    """
    nodes = list(G.nodes)
    if not nodes:
        return {}
    idx = {n: i for i, n in enumerate(nodes)}
    seeds = [e for e in question_entities if e in G.nodes]
    if not seeds:
        return {}

    A = nx.to_numpy_array(G, nodelist=nodes, weight="weight")
    # Normalize rows so high-degree nodes don't dominate
    row_sums = A.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    A = A / row_sums

    seed_vec = np.zeros(len(nodes))
    for s in seeds:
        seed_vec[idx[s]] = 1.0 / len(seeds)

    score_vec = np.zeros(len(nodes))
    current = seed_vec.copy()
    for k in range(1, max_path + 1):
        current = beta * (A.T @ current)
        score_vec += current

    return {nodes[i]: score_vec[i] for i in range(len(nodes))}



In [ ]:
# ── 5. EVALUATION ─────────────────────────────────────────────────────────────

def evaluate(dataset, scorer_fn, scorer_name: str, n_examples: int = 200):
    """
    For each example:
      - positive label: answer entity (and supporting fact entities)
      - negative labels: all other entities in the graph
    We rank all nodes by their score and compute AUC-ROC and Average Precision.
    """
    all_labels, all_scores = [], []

    for example in list(dataset)[:n_examples]:
        G = build_example_graph(example)
        if G.number_of_nodes() < 5:
            continue  # skip degenerate graphs

        question_ents = extract_entities(example["question"])
        answer_ents   = extract_entities(example["answer"])

        # Build ground-truth: nodes that appear in supporting facts are positive
        positive_nodes = set(answer_ents)
        for sent in example["supporting_facts"]["title"]:
            positive_nodes.update(extract_entities(sent))
        positive_nodes = {e.lower().strip() for e in positive_nodes}

        scores = scorer_fn(G, question_ents)
        if not scores:
            continue

        for node, score in scores.items():
            all_scores.append(score)
            all_labels.append(1 if node in positive_nodes else 0)

    if sum(all_labels) == 0:
        print(f"{scorer_name}: no positives found — check entity extraction")
        return

    auc  = roc_auc_score(all_labels, all_scores)
    ap   = average_precision_score(all_labels, all_scores)
    print(f"{scorer_name:20s}  AUC-ROC: {auc:.4f}  Avg Precision: {ap:.4f}")



In [ ]:
# ── 6. RUN EXPERIMENTS ────────────────────────────────────────────────────────

from functools import partial

print("Evaluating on 200 HotpotQA validation examples...\n")
evaluate(dataset, score_candidates_ppr,     "Personalized PageRank")

Evaluating on 200 HotpotQA validation examples...

Personalized PageRank  AUC-ROC: 0.8084  Avg Precision: 0.2228


In [ ]:
evaluate(dataset, score_candidates_jaccard, "Jaccard")


Jaccard               AUC-ROC: 0.6295  Avg Precision: 0.0273


In [ ]:
evaluate(dataset, score_candidates_adamic,  "Adamic-Adar")


Adamic-Adar           AUC-ROC: 0.6158  Avg Precision: 0.0289


In [ ]:
evaluate(dataset, score_candidates_katz,    "Katz")

Katz                  AUC-ROC: 0.7252  Avg Precision: 0.0666
